# SparseChron: 4D Gaussian Splatting on Kaggle

This notebook provides a complete, one-click pipeline to train the **SparseChron** 4D Gaussian Splatting engine on Kaggle using a single T4 GPU.

It will automatically:
1. Clone the necessary repositories (SparseChron, DUSt3R, Depth Anything V2)
2. Download the `interp_cut-lemon` dataset
3. Run DUSt3R for pose estimation and Depth Anything for depth maps
4. **Run a custom back-projection script** to fix depth scaling issues and properly anchor the 4D temporal trajectory.
5. Train the 4D model and output checkpoints.
6. Render a final MP4 video of your 4D hologram.

### 1. Install System Dependencies

In [ ]:
!apt-get update && apt-get install -y git ninja-build libglib2.0-0 libsm6 libxrender-dev libxext6 ffmpeg

### 2. Clone Repositories

In [ ]:
%cd /kaggle/working
!rm -rf SparseChron Depth-Anything-V2 dust3r data interp_cut-lemon

!git clone https://github.com/rajhodedara/SparseChron.git
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
!git clone --recursive https://github.com/naver/dust3r

### 3. Install Python Requirements

In [ ]:
%cd /kaggle/working/SparseChron
!pip install -r requirements.txt

### 4. Download Sample Dataset (Cut Lemon)

In [ ]:
%cd /kaggle/working
!wget https://github.com/google/hypernerf/releases/download/v0.1/interp_cut-lemon.zip
!unzip -q interp_cut-lemon.zip

### 5. Preprocess (Poses & Depth Maps)
*This uses DUSt3R and Depth Anything V2 to guess camera locations and generate depth maps.*

In [ ]:
%cd /kaggle/working/SparseChron
import os
os.environ['PYTHONPATH'] = "/kaggle/working/SparseChron:/kaggle/working/Depth-Anything-V2:/kaggle/working/dust3r:/kaggle/working/dust3r/croco"

!python scripts/preprocess.py --scene-dir /kaggle/working/interp_cut-lemon/rgb/2x --output-dir /kaggle/working/data

### 6. Generate Mathematically Correct Point Cloud
*This custom script back-projects the depth maps to correct DUSt3R's scale mismatch and properly anchors the 4D trajectory.*

In [ ]:
%cd /kaggle/working/SparseChron
!python scripts/generate_init_ply.py --data-dir /kaggle/working/data

### 7. Train the 4D Gaussian Splatting Engine
*This will train the model. It automatically saves checkpoints to `/kaggle/working/outputs/lego_exp1` every 10,000 iterations. Let it run for at least 6,000 to 10,000 iterations!*

In [ ]:
%cd /kaggle/working/SparseChron
!PYTHONPATH=/kaggle/working/SparseChron python scripts/train.py --scene-dir /kaggle/working/data --is-4d --resume-from None

### 8. Render Image Frames
*Once you manually stop the training, run this to render the frames using your best checkpoint.*

In [ ]:
%cd /kaggle/working/SparseChron
# NOTE: Update the checkpoint name below (e.g. checkpoint_10000.ckpt) to match the one saved in your outputs folder!
!PYTHONPATH=/kaggle/working/SparseChron python scripts/evaluate.py --checkpoint-path /kaggle/working/outputs/lego_exp1/checkpoint_10000.ckpt --dataset-path /kaggle/working/data

### 9. Stitch Video
*Combine the rendered PNGs into a smooth MP4 video.*

In [ ]:
%cd /kaggle/working/SparseChron
!ffmpeg -y -framerate 15 -i evaluation_output/renders/render_%05d.png -c:v libx264 -pix_fmt yuv420p final_4D_video.mp4
print("Done! Download final_4D_video.mp4 from the SparseChron folder!")